# 🏫 Proyecto Final: Identificación y Clasificación del Riesgo de Deserción Escolar en Ecuador
**Curso:** Inteligencia Artificial - ESPOL  
**Profesor:** Enrique Peláez J. Ph.D.  
**Grupo #9:** Mateo Mayorga, Anthony Navarrete, Andrés Salinas  
**Dataset:** Ministerio de Educación del Ecuador (MINEDUC) - Datos Abiertos

---  
### 🎯 Correcciones Aplicadas de la Tarea #4:
1. **División Temporal (Time-based Split):** Se entrena con años históricos anteriores y se evalúa sobre el período más reciente para eliminar el sesgo temporal.
2. **Prevención de Data Leakage:** Las características $X$ se construyen estrictamente con variables de **Inicio de Año**. Las variables de **Fin de Año** se usan únicamente para calcular la etiqueta $y$.
3. **Manejo de Desbalance:** Ponderación `class_weight='balanced'` activada en los modelos de comparación.

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.utils.class_weight import compute_class_weight

sys.path.append(os.path.abspath('..'))
from src.data_preprocessing import DataPreprocessor
from src.mlp_classifier import MLPClassifier
from src.baseline_evaluator import BaselineEvaluator
from src.shap_explainer import SHAPExplainer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('✅ Librerías cargadas correctamente.')

In [ ]:
ruta_inicio = '../data/raw/1Registro-Administrativo-Historico_2009-202X-Inicio.xlsx'
ruta_fin = '../data/raw/2Registro-Administrativo-Historico_2009-2024-Fin.xlsx'

preprocesador = DataPreprocessor(ruta_inicio, ruta_fin)
df_raw = preprocesador.cargar_y_fusionar_datasets(sample_size=4000)
df_clean = preprocesador.limpiar_y_calcular_abandono(df_raw)
df_final = preprocesador.discretizar_riesgo(df_clean)

# Matriz X libre de data leakage
X, y = preprocesador.transformar_caracteristicas(df_final, is_training=True)

# Partición cronológica por año lectivo
X_train, X_test, y_train, y_test, info_split = preprocesador.dividir_por_tiempo(df_final, X, y)

print(f'📅 Criterio de Partición: {info_split}')
print(f'📊 Tamaño Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print('🛡️ Variables de Entrada (Primeras 5):', preprocesador.feature_names[:5])

In [ ]:
print('🧠 Entrenando Perceptrón Multicapa (MLP)...')
mlp = MLPClassifier(input_dim=X_train.shape[1], num_classes=3)
history = mlp.entrenar(X_train, y_train, epochs=40, batch_size=32)
y_pred_mlp = mlp.predecir(X_test)
print('✅ Entrenamiento completado.')

In [ ]:
evaluador = BaselineEvaluator()
df_baseline = evaluador.entrenar_y_evaluar_todos(X_train, y_train, X_test, y_test)

acc_mlp = accuracy_score(y_test, y_pred_mlp)
prec_mlp = precision_score(y_test, y_pred_mlp, average='macro', zero_division=0)
rec_mlp = recall_score(y_test, y_pred_mlp, average='macro', zero_division=0)
f1_mlp = f1_score(y_test, y_pred_mlp, average='macro', zero_division=0)

fila_mlp = pd.DataFrame([{
    'Modelo': 'MLP (Propio - Keras)',
    'Accuracy': round(acc_mlp, 4),
    'Precision (Macro)': round(prec_mlp, 4),
    'Recall (Macro)': round(rec_mlp, 4),
    'F1-Score (Macro)': round(f1_mlp, 4)
}])

tabla_final = pd.concat([fila_mlp, df_baseline], ignore_index=True)
display(tabla_final)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

todos_los_modelos = {'MLP (Keras)': y_pred_mlp}
for nombre, modelo in evaluador.modelos_entrenados.items():
    todos_los_modelos[nombre] = modelo.predict(X_test)

clases_labels = ['Bajo (0)', 'Medio (1)', 'Alto (2)']

for idx, (nombre, preds) in enumerate(todos_los_modelos.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], xticklabels=clases_labels, yticklabels=clases_labels)
    axes[idx].set_title(f'Matriz: {nombre}', fontweight='bold')
    axes[idx].set_xlabel('Predicción')
    axes[idx].set_ylabel('Real')

plt.tight_layout()
plt.show()

In [ ]:
print('🔍 Explicabilidad con SHAP sobre la partición temporal...')
explainer = SHAPExplainer(mlp.model.predict, X_train, preprocesador.feature_names)
fig_shap = explainer.generar_grafico_resumen(X_test, n_samples=15)
plt.show()